<a id="top"></a>
# How to Browse Roman Observations with astroquery
***
## Learning Goals

This is a beginner tutorial on browsing Roman Space Telescope observations and data products available on MAST. We'll cover the basics of querying for Roman data and viewing results interactively. By the end of this tutorial, you will:

- Understand how to query MAST for Roman observations using `MastMissions`.
- Be able to view and explore observation tables with `mast-table`.
- Know how to visualize observation footprints with `mast-aladin`.

## Introduction

This tutorial shows you how to search for Roman observations at MAST and explore them interactively in a Jupyter notebook.

You'll use three Python packages:
- [`astroquery.mast.MastMissions`](https://astroquery.readthedocs.io/en/latest/mast/mast_missions.html) to query the MAST Search API for Roman observation metadata.
- [`mast-table`](https://github.com/spacetelescope/mast-table) to view query results in an interactive table.
- [`mast-aladin`](https://github.com/spacetelescope/mast-aladin) to visualize observation footprints on a sky map.

## Imports

We will use the following packages in this tutorial:

- `astroquery.mast.MastMissions`: query MAST for Roman observation metadata.
- `astropy.coordinates.SkyCoord`: express target positions on the sky.
- `astropy.units`: attach physical units (e.g., degrees) to numerical values.
- `mast_table.MastTable`: interactive widget for browsing tabular query results.
- `mast_aladin.MastAladin`: interactive sky map widget for visualizing observation footprints.

In [ ]:
import astropy.units as u
from astropy.coordinates import SkyCoord
from astroquery.mast import MastMissions

from mast_aladin import MastAladin
from mast_table import MastTable

***

## Querying for Roman Observations

In this section, we configure `astroquery` to query MAST for Roman observations and run a few example searches. We use the `MastMissions` class, which is the recommended interface for mission-specific metadata searches.

### Setting Up the Query Connection

We first create a `MastMissions` object configured for Roman, then point its internal connection to the MAST test server (`masttest.stsci.edu`) that hosts the Roman observations.

In [ ]:
# MAST test server endpoints for Roman.
MASTTEST_HOST = "https://masttest.stsci.edu"
MASTTEST_REQUEST_URL = f"{MASTTEST_HOST}/search_sandbox/roman/api/v0.1/"
MASTTEST_DOWNLOAD_URL = f"{MASTTEST_HOST}/search_sandbox/"

# Create a MastMissions object for Roman and point it at the MAST test server.
mast = MastMissions(mission="roman")
mast._service_api_connection.SERVICE_URL = MASTTEST_HOST
mast._service_api_connection.REQUEST_URL = MASTTEST_REQUEST_URL
mast._service_api_connection.MISSIONS_DOWNLOAD_URL = MASTTEST_DOWNLOAD_URL

As a quick connectivity check, we ask the API for the list of searchable columns. This both verifies our connection and gives us a reference of the metadata fields we can filter on.

In [ ]:
# Retrieve the searchable column list for Roman.
columns = mast.get_column_list()
print(f"Found {len(columns)} searchable columns. First 10:\n")
for row in columns[:10]:
    print(f"  {row['name']:30s} ({row['data_type']})")

### Basic Query for Roman Data

`query_region()` searches for observations within a given radius of a sky position. In the example below, we filter by `optical_element='F146'`, the Roman WFI wide-band filter.

In [ ]:
# Define a target position with Roman coverage.
target = SkyCoord(ra=270, dec=66, unit="deg")

# Query for Roman observations within 0.3 deg of the target,
# restricted to the F146 filter.
roman_obs = mast.query_region(
    target,
    radius=0.3 * u.deg,
    optical_element="F146",
)
print(f"Found {len(roman_obs)} Roman observations with F146 filter")
roman_obs[:5]

**Querying across product levels.** If you want a broader view that does not constrain a specific filter, you can use `query_criteria()` with `productLevel` to retrieve all processing levels in the region:

In [ ]:
# Retrieve all processing levels in the same region.
all_obs = mast.query_criteria(
    coordinates=target,
    radius=0.3 * u.deg,
    productLevel=["1", "2", "3", "4"],
)
print(f"Found {len(all_obs)} observations across all processing levels")
all_obs[:3]

The results are returned as an [`astropy.table.Table`](https://docs.astropy.org/en/stable/table/). Useful columns include:

- `fileSetName`: unique identifier for the observation file set.
- `productLevel`: processing level (1=raw, 2=calibrated, 3=resampled, 4=high-level).
- `optical_element`: filter used (F062, F087, F106, F129, F146, F158, F184 for WFI).
- `exposure_time`: exposure time in seconds.
- `detector`: detector identifier (WFI01 to WFI18 for the WFI mosaic).
- `program`: program ID number.
- `s_region`: sky-region footprint as an STC-S `POLYGON` string.

### Filtering by Position

You can run the same kinds of queries at any sky position. The example below queries for Roman observations near the galactic center.

In [ ]:
# Define galactic-center coordinates.
galactic_center = SkyCoord(ra=266.4, dec=-29.0, unit="deg")

# Query Roman observations within 0.5 deg, including all processing levels.
position_obs = mast.query_criteria(
    coordinates=galactic_center,
    radius=0.5 * u.deg,
    productLevel=["1", "2", "3", "4"],
)
print(f"Found {len(position_obs)} observations near the galactic center")
position_obs[:5]

All of the examples above use `query_region()` or `query_criteria()` with a `SkyCoord` object and a radius. You can also use `query_object()` with a target name (resolved internally by `astroquery`). For the full set of query options, see the [`astroquery.mast.MastMissions` documentation](https://astroquery.readthedocs.io/en/latest/mast/mast_missions.html).

***

## Viewing Results with mast-table

`mast-table` is an interactive Jupyter widget for browsing the metadata returned by a MAST query. It supports pagination, column sorting, search-based filtering, and column visibility toggles: all without leaving the notebook.

### Creating an Interactive Table

Pass the Astropy table returned by the query directly to `MastTable`:

In [ ]:
# Create an interactive table widget from our query results.
table_widget = MastTable(roman_obs)
table_widget

The table widget displays above with several interactive features:
- **Pagination controls** at the bottom right to navigate through pages of results.
- **Column sorting**: click on a column header to sort by that column.
- **Search box**: filter rows by typing in the search field.
- **Column visibility**: toggle which columns are displayed.

### Exploring Table Features

Try the following to explore the table's capabilities:

1. **Change pages.** Use the arrow buttons at the bottom right to view different pages of results.
2. **Sort data.** Click on the `optical_element` column header to sort observations by filter.
3. **Search.** Try typing a program ID (for example, `185`) in the search box to filter the table.
4. **Adjust page size.** Use the "Rows per page" dropdown to display more or fewer results at once.

For a deeper dive on customizing `mast-table`, see the companion notebook Inspect Long Tables.

***

## Visualizing Footprints with mast-aladin

For a spatial view of the observations, `mast-aladin` renders an interactive sky map (powered by Aladin Lite) directly in the notebook. This is particularly useful for understanding the sky coverage of Roman observations and how they overlap.

### Loading Footprints

Create an Aladin widget centered on our target region. The sky map can then be overlaid with the polygon footprint of each Roman observation.

In [ ]:
# Create an Aladin widget centered on our target.
aladin = MastAladin(
    target=f"{target.ra.deg} {target.dec.deg}",
    fov=1.0,  # Field of view in degrees.
    height=500,
)
aladin

### Interactive Sky View

The Aladin widget provides several interactive features:

- **Pan.** Click and drag to move around the sky.
- **Zoom.** Use the scroll wheel or the zoom controls.
- **Layers.** Toggle different survey layers (e.g., DSS, 2MASS) using the layer controls.
- **Footprints.** Observation footprints can be displayed as polygons.

Roman footprints are stored in the `s_region` column as STC-S `POLYGON` strings, which `mast-aladin` can overlay directly:

In [ ]:
# Overlay Roman observation footprints on the Aladin sky map.
if len(roman_obs) > 0 and "s_region" in roman_obs.colnames:
    aladin.add_graphic_overlay_from_stcs(
        roman_obs["s_region"],
        color="cyan",
        opacity=0.5,
    )
    print(f"Added {len(roman_obs)} observation footprints to the map")
else:
    print("No footprint data available")

The cyan polygons show the actual sky coverage of each Roman observation in our query. Scroll back to the Aladin widget above to inspect them: you can pan, zoom, and toggle background surveys to see how Roman tiles overlap real reference imagery. For more options, see the [`mast-aladin` documentation](https://github.com/spacetelescope/mast-aladin).

***

## Additional Resources

- [Roman Space Telescope mission site](https://roman.gsfc.nasa.gov/): official mission information.
- [MAST Roman page](https://archive.stsci.edu/missions-and-data/roman): archive landing page for Roman.
- [`astroquery.mast.MastMissions` documentation](https://astroquery.readthedocs.io/en/latest/mast/mast_missions.html): API reference for mission-specific MAST queries.
- [`mast-table` repository](https://github.com/spacetelescope/mast-table): widget documentation and examples.
- [`mast-aladin` repository](https://github.com/spacetelescope/mast-aladin): interactive sky-visualization documentation.
- [MAST Search API](https://mast.stsci.edu/search/docs/): underlying REST API used by `MastMissions`.
- Companion notebook: Inspect Long Tables: deeper dive into customizing `mast-table` views.

## Citations

If you use `astroquery`, `astropy`, `mast-table`, or `mast-aladin` for published research, please cite the authors. Follow these links for more information:

- [Citing `astroquery`](https://github.com/astropy/astroquery/blob/main/astroquery/CITATION)
- [Citing `astropy`](https://www.astropy.org/acknowledging.html)
- [Citing `mast-table`](https://github.com/spacetelescope/mast-table)
- [Citing `mast-aladin`](https://github.com/spacetelescope/mast-aladin)

## About This Notebook

**Author:** Hatice Karatay <br>
**Keywords:** Roman, MAST, astroquery, mast-table, mast-aladin, observations <br>
**First published:** June 2026 <br>
**Last updated:** June 2026 <br>

***
[Top of Page](#top)
<img style="float: right;" src="https://raw.githubusercontent.com/spacetelescope/style-guides/master/guides/images/stsci-logo.png" alt="Space Telescope Logo" width="200px"/>